# Attention & Decoding

In [ ]:
%pip -q install "transformers==4.57.1" sentencepiece sacremoses

In [ ]:
import pandas as pd
import torch
from IPython.display import display
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, set_seed

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## Task 1 · Attention in Korean-to-English Translation

Start with one sentence.

> 저는 월요일에 학교에 갑니다.

Each row shows the attention weights for one English token. The columns show the Korean tokens. Darker cells show higher weights. **End** marks the end of the Korean sentence.

In [ ]:
TRANSLATION_MODEL_ID = "Helsinki-NLP/opus-mt-ko-en"
translation_tokenizer = AutoTokenizer.from_pretrained(TRANSLATION_MODEL_ID, use_fast=False)
translator = AutoModelForSeq2SeqLM.from_pretrained(
    TRANSLATION_MODEL_ID, use_safetensors=False, attn_implementation="eager"
).to(device).eval()

print(f"Layers: 0–{translator.config.decoder_layers - 1}")
print(f"Heads: 0–{translator.config.decoder_attention_heads - 1}")

In [ ]:
sentence = "저는 월요일에 학교에 갑니다."
layer = 1
head = 4

source = translation_tokenizer(sentence, return_tensors="pt").to(device)
source_ids = source.input_ids[0].tolist()
if translation_tokenizer.unk_token_id in source_ids:
    raise ValueError("An unknown Korean token was found. Try a more common word.")

with torch.inference_mode():
    result = translator.generate(
        **source,
        do_sample=False,
        num_beams=1,
        max_new_tokens=24,
        return_dict_in_generate=True,
        output_attentions=True,
    )

# Each generated token has one attention step. Skip the decoder start token.
generated_ids = result.sequences[0, 1:].tolist()
translation = translation_tokenizer.decode(generated_ids, skip_special_tokens=True)
print(f"Korean: {sentence}")
print(f"English: {translation}")

source_tokens = translation_tokenizer.convert_ids_to_tokens(source_ids)
target_tokens = translation_tokenizer.convert_ids_to_tokens(generated_ids)
source_labels = [
    "End" if token == translation_tokenizer.eos_token else token.removeprefix("▁") or "Space"
    for token in source_tokens
]
columns = [f"{i + 1}. {label}" for i, label in enumerate(source_labels)]
special_tokens = set(translation_tokenizer.all_special_tokens)
rows = []
weights = []
for i, (token, step) in enumerate(zip(target_tokens, result.cross_attentions)):
    if token not in special_tokens:
        rows.append(f"{i + 1}. {token.removeprefix('▁') or 'Space'}")
        weights.append(step[layer][0, head, -1, :].tolist())

table = pd.DataFrame(weights, index=rows, columns=columns)
table.index.name = "English token"
display(
    table.style
    .set_caption(f"Layer {layer}, head {head}")
    .format(precision=3)
    .format_index(escape="html", axis=0)
    .format_index(escape="html", axis=1)
    .background_gradient(cmap="YlGnBu", axis=None, vmin=0, vmax=1)
    .set_properties(**{"white-space": "nowrap", "text-align": "center"})
)

### Try it yourself

1. **Change the day.** In `sentence`, replace `월요일` with another day. Run the cell. How did the English day change? Find the row for the new day. Which Korean token gets the most attention?
2. **Change the place.** Start with the original sentence again. Replace `학교` with another place and keep the day the same. Run the cell. Find the row for the new place in English. Which Korean token gets the most attention?
3. **Change the head and layer.** Use the original sentence again. Keep `layer = 1` and change `head` from 4 to 5. Then keep `head = 5` and change `layer` from 1 to 0. Run the cell after each change. In the **Monday** row, does the same Korean token still get the most attention?

## Task 2 · Decoding

### A. Summarize a notice

Use FLAN-T5-small to write a one-sentence summary of the workshop notice. Keep the notice, prompt, and maximum length the same. Change only the decoding method.

### Workshop notice

> The campus library will hold an academic writing workshop on Friday. The workshop was originally scheduled for 2 p.m., but it will now start at 4 p.m. It will take place in Room 204 of the library. Anyone planning to attend must register online by noon on Thursday. Admission is free, and students do not need to bring a laptop. A recording will be available the following week.

### Question

Write your own one-sentence summary before you look at the model's answer. How do you think your summary will differ from the model's?

### Compare with Greedy

Read the model's summary below. Which details did each summary keep or leave out?

In [1]:
SUMMARY_MODEL_ID = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(SUMMARY_MODEL_ID, use_fast=True)
model = AutoModelForSeq2SeqLM.from_pretrained(
    SUMMARY_MODEL_ID, use_safetensors=True
).to(device).eval()

workshop_notice = (
    "The campus library will hold an academic writing workshop on Friday. The workshop was "
    "originally scheduled for 2 p.m., but it will now start at 4 p.m. It will take place in "
    "Room 204 of the library. Anyone planning to attend must register online by noon on "
    "Thursday. Admission is free, and students do not need to bring a laptop. A recording "
    "will be available the following week."
)
summary_prefix = "Summarize this notice in one sentence: "
MAX_NEW_TOKENS = 64
SEED = 42
GREEDY = {"do_sample": False, "num_beams": 1}
BEAM_SEARCH = {"do_sample": False, "num_beams": 4}
RANDOM_SAMPLING = {
    "do_sample": True, "num_beams": 1, "temperature": 1.0, "top_k": 0, "top_p": 1.0
}
TEMPERATURE_LOW = {**RANDOM_SAMPLING, "temperature": 0.7}
TEMPERATURE_HIGH = {**RANDOM_SAMPLING, "temperature": 1.3}
TOP_K = {**RANDOM_SAMPLING, "top_k": 20}
TOP_P = {**RANDOM_SAMPLING, "top_p": 0.9}

def generate_text(prompt, **options):
    source = tokenizer(prompt, return_tensors="pt").to(device)
    if options["do_sample"]:
        set_seed(SEED)
    with torch.inference_mode():
        ids = model.generate(**source, max_new_tokens=MAX_NEW_TOKENS, **options)
    return tokenizer.decode(ids[0], skip_special_tokens=True)


workshop_prompt = summary_prefix + workshop_notice
workshop_greedy = generate_text(workshop_prompt, **GREEDY)
print("Greedy summary")
print(workshop_greedy)

Greedy summary
The campus library will hold an academic writing workshop on Friday.


### Check the Greedy summary

| What to check | What we found |
| --- | --- |
| **Key facts kept (0–3)** | **0/3.** It says Friday but leaves out the new start time, 4 p.m. It also leaves out Room 204 and the sign-up deadline. |
| **Wrong or added facts** | None. The facts it gives match the notice. |
| **Repeated or awkward wording** | No repetition. The sentence reads naturally. |

### Try it yourself

1. **Write your own code.** Use `generate_text()` with `workshop_prompt` and the settings above to try Beam Search, Random Sampling, Temperature Sampling, Top-k, and Top-p. Run each method once. The helper uses seed 42 for sampling.
2. **Check each summary.** Give one point for each key fact it keeps. Also record any wrong or added facts, repetition, or awkward wording.
3. **Choose a method.** Which summary would you share with a classmate, and why?

The three key facts are **Friday at 4 p.m.**, **Room 204**, and **the sign-up deadline at noon on Thursday**.

In [ ]:
# Write your code here.

### B. Continue a story

> **Imagine this:** Elon Musk lands on Mars. He sees footprints near his rocket.

**What happens next? Continue the story in one sentence.**

1. Use the same `story_prompt` with `GREEDY`, `BEAM_SEARCH`, and `TOP_P`. Run each method once. Use the same maximum length and seed 42 for Top-p.
2. Read each result. Does it repeat the opening? Does it change a detail, such as the setting? Does it add a new event that fits the story?
3. Would you choose the same decoding method for the workshop summary and the story? Explain your choice using one detail from each task.

In [ ]:
story_prompt = "Imagine this: Elon Musk lands on Mars. He sees footprints near his rocket. What happens next? Continue the story in one sentence."

# Write your code here.

[Korean-to-English model](https://huggingface.co/Helsinki-NLP/opus-mt-ko-en) · [FLAN-T5-small model](https://huggingface.co/google/flan-t5-small) · [Decoding settings](https://huggingface.co/docs/transformers/main_classes/text_generation)